In [5]:

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():  # handles running from the notebooks/ folder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [6]:
from src.metrics import evaluate, confusion, save_preds
from src.data import load_splits

In [9]:
import pandas as pd
import numpy as np
from src.metrics import evaluate, confusion, save_preds
from src.data import load_splits
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
DATA, PRED, RES = ROOT / "data", ROOT / "predictions", ROOT / "results"


S = load_splits(DATA)
# test is NOT loaded into a variable on purpose: it stays sealed
train, val = S["train"], S["val"]
y_tr, y_va = train.rating.values, val.rating.values
# one dict per model, turned into a table at the end
results = []
print(len(train), len(val))




42681 4743


In [10]:
from sklearn.dummy import DummyClassifier
m = DummyClassifier(strategy="most_frequent").fit(
    train[["norm"]], y_tr)   # learns only "most frequent class = 5"
p = m.predict(val[["norm"]])
results.append(evaluate(y_va, p, "A1_majority"))
save_preds(val, p, "A1_majority", PRED)
results[-1]

{'model': 'A1_majority', 'macro_f1': 0.1084, 'accuracy': 0.3717, 'mae': 1.1324}

In [12]:
from sklearn.linear_model import LogisticRegression


def len_feats(d):
    n = d.text.str.split().str.len()
    # log: 10→20 words matters more than 1000→1010
    return np.c_[np.log1p(n), np.log1p(d.text.str.len())]


m = LogisticRegression(class_weight="balanced",
                       max_iter=1000).fit(len_feats(train), y_tr)
p = m.predict(len_feats(val))
results.append(evaluate(y_va, p, "A2_length"))
save_preds(val, p, "A2_length", PRED)
results[-1]

{'model': 'A2_length', 'macro_f1': 0.1606, 'accuracy': 0.183, 'mae': 1.7729}

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec_w = TfidfVectorizer(ngram_range=(1, 2),   # single words + pairs ("غير جيد")
                        min_df=2,             # ignore words seen only once: typos, noise
                        sublinear_tf=True)    # 1+log(tf): a word repeated 10x is not 10x more important
# FIT on train only: vocabulary and IDF come from train
Xw_tr = vec_w.fit_transform(train.norm)
# val is only transformed with the train vocabulary
Xw_va = vec_w.transform(val.norm)
print("vocab size:", Xw_tr.shape[1])

for C in [0.3, 1, 3, 10]:
    m = LogisticRegression(C=C, class_weight="balanced",
                           max_iter=2000).fit(Xw_tr, y_tr)
    print(C, evaluate(y_va, m.predict(Xw_va), f"C={C}"))

vocab size: 329379
0.3 {'model': 'C=0.3', 'macro_f1': 0.4112, 'accuracy': 0.4617, 'mae': 0.7565}
1 {'model': 'C=1', 'macro_f1': 0.4143, 'accuracy': 0.4672, 'mae': 0.719}
3 {'model': 'C=3', 'macro_f1': 0.4098, 'accuracy': 0.4638, 'mae': 0.708}
10 {'model': 'C=10', 'macro_f1': 0.405, 'accuracy': 0.4622, 'mae': 0.7046}


In [14]:
from scipy.sparse import hstack
vec_c = TfidfVectorizer(analyzer="char_wb",        # char n-grams inside word boundaries
                        ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000)
Xc_tr, Xc_va = vec_c.fit_transform(train.norm), vec_c.transform(val.norm)
X_tr, X_va = hstack([Xw_tr, Xc_tr]).tocsr(), hstack(
    [Xw_va, Xc_va]).tocsr()   # glue both feature sets side by side

BEST_C = 1   # <- replace with the best C from Cell 4
m_lr = LogisticRegression(
    C=BEST_C, class_weight="balanced", max_iter=2000).fit(X_tr, y_tr)
p = m_lr.predict(X_va)
results.append(evaluate(y_va, p, "A3_tfidf_word+char_LR"))
save_preds(val, p, "A3_tfidf_LR", PRED)
results[-1]

{'model': 'A3_tfidf_word+char_LR',
 'macro_f1': 0.4233,
 'accuracy': 0.4681,
 'mae': 0.7063}

In [15]:
from sklearn.linear_model import Ridge
for a in [1, 3, 10]:
    # alpha = regularization (like 1/C)
    r = Ridge(alpha=a).fit(X_tr, y_tr)
    p = np.clip(np.rint(r.predict(X_va)), 1, 5).astype(
        int)   # 4.3 -> 4, and keep inside 1..5
    print(a, evaluate(y_va, p, f"ridge a={a}"))

1 {'model': 'ridge a=1', 'macro_f1': 0.3398, 'accuracy': 0.4267, 'mae': 0.6804}
3 {'model': 'ridge a=3', 'macro_f1': 0.3096, 'accuracy': 0.416, 'mae': 0.6814}
10 {'model': 'ridge a=10', 'macro_f1': 0.2727, 'accuracy': 0.4006, 'mae': 0.6953}


In [16]:
confusion(y_va, m_lr.predict(X_va))

,pred 1,pred 2,pred 3,pred 4,pred 5
true 1,99,66,31,11,15
true 2,62,131,136,34,34
true 3,38,135,362,253,143
true 4,25,78,327,524,476
true 5,27,30,177,425,1104


In [17]:
names = np.r_[vec_w.get_feature_names_out(), vec_c.get_feature_names_out()]
for k, cls in enumerate(m_lr.classes_):
    if cls in (1, 3, 5):
        # 12 largest weights for this class
        top = np.argsort(m_lr.coef_[k])[-12:][::-1]
        print(cls, "★:", " | ".join(names[top]))

1 ★: لم | لا انصح | نجمه واحده | نجمه | لم يعجبني | اي | اسوا | لا | نجمه | ممل | نجمه  | النجمه
3 ★: كتاب لطيف | جيد | لكن | لكن | من كده | ليست سيئه | كتاب مسلي | ولكن | بس |  لكن | خفيف |  جي
5 ★: رائع | هذا الكتاب | من اجمل | ائع | ئع | من اروع | اروع | رائع | جميله جدا | روع | رائعه | ما قرات


In [18]:
res = pd.DataFrame(results)
RES.mkdir(exist_ok=True)
res.to_csv(RES / "phaseA_val.csv", index=False)
res

,model,macro_f1,accuracy,mae
0,A1_majority,0.1084,0.3717,1.1324
1,A2_length,0.1606,0.1830,1.7729
2,A2_length,0.1606,0.1830,1.7729
3,A3_tfidf_word+char_LR,0.4233,0.4681,0.7063
